# Eksempel på sesongjustering for MNR
Denne notebooken viser hvordan nasjonalregnskapet sesongjusterer MNR ved hjelp av US census x13. Det er ingen ekte spc filer eller data fra nasjonalregnskapet i dette repoet, istedenfor har vi hentet ut NACE 47.1 ujustert for å demonstrere hvordan sesongjusteringen gjøres.

In [ ]:
# Python pakke for s210/nasjonalregnskapet hvor sesongjusteringen ligger
from nr_utils.sesongjustering import x13
#Andre python pakker denne notebooken er avhengig av.
import requests
import pandas as pd
import plotly.express as px
from pyjstat import pyjstat

Henter data fra ssb gjennom API. henter tabell 07129 ujustert volum for 47.1 Butikkhandel med bredt vareutvalg.

In [ ]:
url = "https://data.ssb.no/api/pxwebapi/v2/tables/07129/data"
params = {
    "lang": "no",
    "outputFormat": "json-stat2",
    "valuecodes[Tid]": "*",
    "valuecodes[ContentsCode]": "VolumUjustert",
    "valuecodes[NACE]": "47.1",
    "heading": "ContentsCode,NACE",
    "stub": "Tid",
}

resp = requests.get(url, params=params)

Noen små steg med databehandling for å få datene på en måte som gjør at vi kan bruke dem videre.

In [ ]:
data = resp.json()
dataset = pyjstat.Dataset.read(resp.text)
df = dataset.write("dataframe")
df.index = df["måned"].str.replace("M","-").to_list()
df = df[["value"]]
df.columns = ["47.1.u"]

df

Vi kan også lage en figur av dataene for å se på sesongmønsteret. 47.1 har en tydelig julehandel topp for eksempel. Det ønsker vi nok å sesongjustere bort så vi kan fokusere på den faktiske nyheten.

In [ ]:
px.line(df, x=df.index, y="47.1.u")

# Sesongjustering

In [ ]:
help(x13.run_x13_from_df)

Når vi kjører help() kan vi se at funksjonen krever flere ting. Det er en grun til at hvert argument er der.
* df: Alle seriene ujustert som vi ønsker justert.
* spec_folder: En filsti til mappen som inneholder spc filene til sesongjusteringsprogrammet. Hver serie må ha sin egen spesifikasjon.
* html: I praksis en dummy hvor man sier ja eller nei(True or False) html format på output fra x13
* outdir: Filsti for output fra x13. Du kan få kvalitetsrapporter om sesongjusteringen. Ved ingen filsti, blir ikke kvalitetsrapportene tatt vare på.

Vi har allerede en pandas dataramme, men trenger en spc file for å sesongjustere serien vi kan lage spec filer med å følge dokumentasjonen til US census.
* [Dokumentasjon](https://www2.census.gov/software/x-13arima-seats/x13as/windows/documentation/gettingstartedx13-winx13.pdf)

Basert på syntaksen kan vi lage en veldig enkel sesongjustering for 47.1.u hvor vi bruker en airline-model (0,1,1)(0,1,1). Vi lagrer output save d11, d12 og d13. Disse står for sesongjustert, trend og ireggulær. Vi bruker også helt vanlig x11 modus.

In [ ]:
spc = """
Series {
    title="47.1.u"
    period=12
    start=2000.1
}
Arima {
    model=(0,1,1)(0,1,1)
}
x11{
	mode=mult
	seasonalma=x11default
	save=(d11 d12 d13)
}
"""

with open("test_spc/47.1.u.spc","w") as f:
    f.write(spc)

In [ ]:
x13.run_x13_from_df(
    df =df,
    spec_folder="test_spc/")